# SWE-Finetune: Multi-Phase Training for SWE-bench & TerminalBench

This notebook trains Qwen3-30B-A3B using Tinker API for optimal performance on:
- **SWE-bench**: Software engineering agent tasks
- **TerminalBench**: Terminal/shell command tasks

## Phases
1. **Coding Foundation** - Magicoder, Evol-Instruct (~155K)
2. **Terminal/Shell** - NL-SHELL-MULTI, NL2SH-ALFA (~145K)
3. **Tool-Use** - xLAM, Glaive function calling (~180K)
4. **SWE-bench Trajectories** - Agent traces (~156K) **CRITICAL**
5. **Competitive Programming** - TACO, CodeForces (~35K)

## 1. Setup

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q tinker tinker-cookbook datasets transformers

In [ ]:
# Clone the repo
!git clone https://github.com/micic-mihajlo/swe-finetune.git /content/swe-finetune 2>/dev/null || echo 'Repo already exists'
import sys
sys.path.insert(0, '/content/swe-finetune')

In [ ]:
# Set Tinker API key
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## 2. Configuration

In [ ]:
#@title Select Training Phase
PHASE = "ALL - Run 1 through 5"  #@param ["1 - Coding Foundation", "2 - Terminal/Shell", "3 - Tool-Use", "4 - SWE-bench (CRITICAL)", "5 - Competitive Programming", "ALL - Run 1 through 5"]
RESUME_FROM_CHECKPOINT = True  #@param {type:"boolean"}
MAX_SAMPLES_PER_DATASET = None  #@param {type:"raw"}

# Parse phase selection
if PHASE.startswith("ALL"):
    PHASES_TO_RUN = [1, 2, 3, 4, 5]
    print("Will run ALL phases sequentially: 1 -> 2 -> 3 -> 4 -> 5")
else:
    PHASES_TO_RUN = [int(PHASE[0])]
    print(f"Will run Phase {PHASES_TO_RUN[0]}: {PHASE}")

In [ ]:
# Load all configs
from configs import (
    phase1_config, phase2_config, phase3_config, 
    phase4_config, phase5_config
)

CONFIGS = {
    1: phase1_config,
    2: phase2_config,
    3: phase3_config,
    4: phase4_config,
    5: phase5_config,
}

print(f"Phases to run: {PHASES_TO_RUN}")
for p in PHASES_TO_RUN:
    c = CONFIGS[p]
    print(f"  Phase {p}: LR={c.training.learning_rate}, Batch={c.training.batch_size}, MaxLen={c.training.max_length}")

## 3. Initialize Tinker

In [ ]:
import tinker
from scripts.training import create_training_client, train_phase
from scripts.preprocessing import get_tokenizer_and_renderer
from scripts.training.utils import find_latest_checkpoint, save_to_drive
from scripts.data_loaders import (
    load_coding_datasets, load_terminal_datasets, load_tooluse_datasets,
    load_swebench_trajectories, load_competitive_datasets,
)

LOADERS = {
    1: load_coding_datasets,
    2: load_terminal_datasets,
    3: load_tooluse_datasets,
    4: load_swebench_trajectories,
    5: load_competitive_datasets,
}

# Will hold training client across phases (reuse LoRA weights)
training_client = None

In [ ]:
# Main training loop - runs all selected phases
for phase_num in PHASES_TO_RUN:
    config = CONFIGS[phase_num]
    print(f"\n{'='*60}")
    print(f"PHASE {phase_num}: {config.name}")
    print(f"{'='*60}")
    
    # Create or reuse training client
    if training_client is None:
        training_client = create_training_client(
            model_name=config.model.name,
            lora_rank=config.model.lora_rank,
        )
        print(f"Created training client for {config.model.name}")
    
    # Get tokenizer and renderer for this phase's max_length
    tokenizer, renderer = get_tokenizer_and_renderer(
        model_name=config.model.name,
        max_length=config.training.max_length,
    )
    
    # Check for checkpoint
    checkpoint_dir = f"/content/drive/MyDrive/swe-finetune/checkpoints/{config.name}"
    if RESUME_FROM_CHECKPOINT:
        checkpoint = find_latest_checkpoint(checkpoint_dir)
        if checkpoint:
            print(f"Resuming from: {checkpoint}")
            training_client.load_state_with_optimizer(checkpoint)
    
    # Load data
    print(f"Loading data...")
    data_iterator = LOADERS[phase_num](
        streaming=True,
        max_samples_per_dataset=MAX_SAMPLES_PER_DATASET,
        shuffle=True,
    )
    
    # Checkpoint callback
    def on_checkpoint(step, path, cfg=config):
        save_to_drive(path, f"swe-finetune/checkpoints/{cfg.name}/step_{step:06d}")
    
    # Training config
    train_config = {
        "learning_rate": config.training.learning_rate,
        "batch_size": config.training.batch_size,
        "checkpoint_every": config.training.checkpoint_every,
        "lr_schedule": config.training.lr_schedule,
        "warmup_steps": config.training.warmup_steps,
        "train_on_what": "last" if phase_num == 4 else "all",
    }
    
    # Train
    print(f"Training with config: {train_config}")
    results = train_phase(
        training_client=training_client,
        data_iterator=data_iterator,
        renderer=renderer,
        config=train_config,
        checkpoint_callback=on_checkpoint,
    )
    
    print(f"Phase {phase_num} complete! Steps: {results['steps']}, Avg Loss: {results['avg_loss']:.4f}")
    
    # Save phase checkpoint
    phase_final = training_client.save_state(name=f"{config.name}-final").result().path
    save_to_drive(phase_final, f"swe-finetune/checkpoints/{config.name}/final")
    print(f"Saved: {config.name}-final")

print(f"\n{'='*60}")
print("ALL PHASES COMPLETE!")
print(f"{'='*60}")

In [ ]:
# Save final model for inference
sampling_client = training_client.save_weights_and_get_sampling_client(name="swe-finetune-final")
print("Final model saved for inference: swe-finetune-final")

In [ ]:
# Test inference
from tinker.types import SamplingParams, ModelInput

test_prompt = "Write a bash command to find all Python files larger than 1MB"
messages = [{"role": "user", "content": test_prompt}]
prompt_chunk = renderer.build_generation_prompt(messages)

result = sampling_client.sample(
    prompt=ModelInput([prompt_chunk]),
    sampling_params=SamplingParams(max_tokens=200, temperature=0.7, stop=renderer.get_stop_sequences()),
    num_samples=1,
)

print("Prompt:", test_prompt)
print("\nGenerated:")
print(tokenizer.decode(result.sequences[0].tokens))